# OpenCV Basics with the UGOT Robot

This notebook introduces two fundamental computer-vision techniques using the UGOT robot's camera and the OpenCV library:

1. **Color Picker** - read the HSV color value at the center of the live camera frame so you can calibrate color-detection ranges.
2. **Bounding Box** - detect a specific color by HSV range and draw a bounding box around it in real time.

Both tools display live video directly inside the notebook using Jupyter's `display` / `handle.update()` pattern.

> **Prerequisites:** Make sure the UGOT robot is powered on and connected to the same network as your computer before running any cell.

## Setup — Connect to the UGOT

Run this cell first to create the robot controller object and connect to the robot over WiFi.

- `ugot.UGOT()` creates the controller.
- `got.initialize(IP)` opens the connection using the robot's IP address. Change the string below if your robot is on a different address.

> If you restart the kernel at any point, re-run this cell before anything else.

In [ ]:
from ugot import ugot

# Create the robot controller object
got = ugot.UGOT()

# Connect to the robot — change this IP if needed
got.initialize("192.168.1.10")

---
# Color Picker

Before you can detect a color reliably, you need to know its **HSV (Hue, Saturation, Value)** range as seen through the robot's specific camera.

This tool shows a live video feed for 10 seconds. A crosshair is drawn at the center of the frame, and the HSV values of that exact center pixel are printed on screen in real time.

**How to use it:**
1. Run the cell.
2. Hold the object you want to detect in front of the camera so it fills the center crosshair.
3. Read off the H, S, and V numbers; these become the basis for your `lower_color` and `upper_color` bounds in the Bounding Box cell below.

In [ ]:
import time
import numpy as np
import cv2
from IPython.display import display, Image

# Open the camera stream on the robot
got.open_camera()

# --- Jupyter live-display setup ---
# Create a blank placeholder image so Jupyter has a display handle to update in-place.
# Without display_id=True every frame would print as a new image, flooding the output.
dummy = np.zeros((240, 320, 3), np.uint8)
ok, jpg = cv2.imencode(".jpg", dummy)
handle = display(Image(data=jpg.tobytes()), display_id=True)

# --- Timing ---
start_time = time.time()
duration = 10  # seconds to run the live preview

while time.time() - start_time < duration:
    # Read a raw JPEG frame from the robot camera
    picture_raw = got.read_camera_data()
    if picture_raw is None:
        continue  # Skip if no frame arrived yet

    # Decode: raw bytes → 1-D NumPy array → BGR color image
    picture_1D = np.frombuffer(picture_raw, np.uint8)
    picture_bgr = cv2.imdecode(picture_1D, cv2.IMREAD_COLOR)
    if picture_bgr is None:
        continue  # Skip if decoding failed

    # Convert BGR → HSV so we can read meaningful color channels
    picture_hsv = cv2.cvtColor(picture_bgr, cv2.COLOR_BGR2HSV)

    # Find the exact center pixel coordinates
    height = picture_hsv.shape[0]
    width = picture_hsv.shape[1]
    center_x = width // 2
    center_y = height // 2

    # Sample the HSV value of the center pixel
    # shape is (rows, cols, channels) so indexing is [row, col, channel]
    h = int(picture_hsv[center_y, center_x, 0])  # Hue:        0–179 in OpenCV
    s = int(picture_hsv[center_y, center_x, 1])  # Saturation: 0–255
    v = int(picture_hsv[center_y, center_x, 2])  # Value:      0–255

    # Draw a crosshair at the center of the frame so you know exactly what pixel is being sampled
    cv2.circle(picture_bgr, (center_x, center_y), 8, (0, 255, 0), 2)  # outer ring
    cv2.line(
        picture_bgr,
        (center_x - 15, center_y),
        (center_x + 15, center_y),
        (0, 255, 0),
        1,
    )  # horizontal bar
    cv2.line(
        picture_bgr,
        (center_x, center_y - 15),
        (center_x, center_y + 15),
        (0, 255, 0),
        1,
    )  # vertical bar

    # Overlay the live HSV reading in the top-left corner
    text = f"H={h}  S={s}  V={v}"
    cv2.putText(
        picture_bgr, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
    )

    # Show a countdown so you know how long is left
    time_left = duration - int(time.time() - start_time)
    cv2.putText(
        picture_bgr,
        f"Time left: {time_left}s",
        (10, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 0, 255),
        2,
    )

    # Re-encode the annotated frame as JPEG and push it to the existing display handle
    ok, jpg = cv2.imencode(".jpg", picture_bgr)
    if ok:
        handle.update(Image(data=jpg.tobytes()))

    time.sleep(0.05)  # ~20 fps; increase to reduce CPU load

print("Finished live color picker.")

---
# Bounding Box

Now that you know the HSV values of your target color (from the Color Picker above), this cell uses them to **detect and highlight that color in real time**.

**How it works:**
1. Each frame is converted to HSV.
2. `cv2.inRange()` creates a **binary mask** — white pixels where the color falls inside `[lower_color, upper_color]`, black everywhere else.
3. `cv2.findContours()` finds connected white regions in the mask.
4. For any region larger than 1,000 pixels (to filter out noise), a **green bounding box** is drawn on the original frame, along with a center dot and a text label showing the box dimensions.

**Tuning tips:**
- Adjust `lower_color` and `upper_color` based on the H/S/V values you read from the Color Picker.
- The `area > 1000` threshold filters small noise blobs — raise it if you're getting spurious detections, lower it for small targets.
- The `duration` variable controls how many seconds the preview runs.

In [ ]:
import time
import numpy as np
import cv2
from IPython.display import display, Image

# Open the camera stream
got.open_camera()
print("The camera is on.")

# Grab one frame to initialise the Jupyter display handle
picture_raw = got.read_camera_data()
if picture_raw is None:
    raise Exception("No camera data — is the robot connected?")

# Decode the initial frame
picture_1D = np.frombuffer(picture_raw, np.uint8)
picture_bgr = cv2.imdecode(picture_1D, cv2.IMREAD_COLOR)
if picture_bgr is None:
    raise Exception("Frame decode failed — check the camera.")

# Create the Jupyter live-display handle (same in-place update trick as the Color Picker)
ok, picture_jpg = cv2.imencode(".jpg", picture_bgr)
handle = display(Image(picture_jpg.tobytes()), display_id=True)

# --- Color range to detect (HSV) ---
# Replace these values with the H, S, V numbers you found with the Color Picker.
# OpenCV HSV ranges: H 0–179, S 0–255, V 0–255
lower_color = np.array([100, 200, 90])  # lower bound: [H_min, S_min, V_min]
upper_color = np.array([200, 255, 255])  # upper bound: [H_max, S_max, V_max]

# --- Timing ---
start_time = time.time()
duration = 10  # seconds to run

while time.time() - start_time < duration:
    # Grab the latest frame
    picture_raw = got.read_camera_data()
    if picture_raw is None:
        continue

    # Decode raw bytes → BGR image
    picture_1D = np.frombuffer(picture_raw, np.uint8)
    picture_bgr = cv2.imdecode(picture_1D, cv2.IMREAD_COLOR)
    if picture_bgr is None:
        continue

    # Convert to HSV for color-range detection
    picture_hsv = cv2.cvtColor(picture_bgr, cv2.COLOR_BGR2HSV)

    # Build a binary mask: 255 where pixel is inside the color range, 0 elsewhere
    mask = cv2.inRange(picture_hsv, lower_color, upper_color)

    # Find the outlines of white (detected) regions in the mask.
    # RETR_EXTERNAL: only outermost contours (ignores holes inside shapes).
    # CHAIN_APPROX_SIMPLE: compress straight-line segments to save memory.
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        area = cv2.contourArea(contour)

        # Ignore small blobs that are likely noise
        if area > 1000:
            # Get the axis-aligned bounding rectangle: (x, y) = top-left corner, w = width, h = height
            x, y, w, h = cv2.boundingRect(contour)

            # Compute the horizontal center of the bounding box
            # (useful later for steering the robot toward the detected object)
            center_x = x + w // 2

            # Draw the green bounding box around the detected region
            cv2.rectangle(picture_bgr, (x, y), (x + w, y + h), (0, 255, 0), 2)

            # Mark the center of the box with a filled blue dot
            cv2.circle(picture_bgr, (center_x, y + h // 2), 5, (255, 0, 0), -1)

            # Label the box with its center x and dimensions
            text = f"cx:{center_x}, w:{w}, h:{h}"
            text_y = max(30, y - 10)  # clamp so text doesn't go off the top edge
            cv2.putText(
                picture_bgr,
                text,
                (x, text_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2,
            )

    # Re-encode and push the annotated frame to the Jupyter display
    ok, picture_jpg = cv2.imencode(".jpg", picture_bgr)
    if ok:
        handle.update(Image(picture_jpg.tobytes()))

    time.sleep(0.1)  # ~10 fps